Dynamic QMDP circuit Two State Transitions with Quantum Return Calculation on ***IBM Quantum Computer***


In [ ]:
# Qiskit Runtime
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService(
    channel="",
    token="",
    instance=""
)

backend = service.backend('ibm_torino')
print(f">>> Connected to {backend.name} backend.")

In [ ]:
# importing packages
from qiskit import QuantumRegister, QuantumCircuit, ClassicalRegister, transpile
from qiskit_aer.backends import AerSimulator
from qiskit.circuit.library import RYGate
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from typing import List

# functions
def reverse_function(x):
    return x[::-1]

def _state_to_latex_ket(
    data: List[complex], max_size: int = 12, prefix: str = "", decimals: int = 10 
    ) :
    num = int(np.log2(len(data)))

    def ket_name(i):
        return bin(i)[2:].zfill(num)

    data = np.around(data, decimals)
    nonzero_indices = np.where(data != 0)[0].tolist()
    latex_terms = data[nonzero_indices] ### changes here 

    nextState_prop_dict= {}
    nextState_bits_dict = {}
    counter = 0
    
    for idx, ket_idx in enumerate(nonzero_indices):
            term = latex_terms[idx]
            ket = ket_name(ket_idx)
            
            # nextState_prop_dict[counter] = np.abs(np.real(term))
            nextState_prop_dict[counter] = (np.real(term))
            nextState_bits_dict[counter] = ket
            counter +=1
            
    return nextState_prop_dict,nextState_bits_dict

In [ ]:
# Classical environment

class ClassicToyEnv:

    def __init__(self, MaxSteps=np.inf):
        
        self.MaxSteps= MaxSteps # Maximum number of steps before ending an episode
        
        # ========================================
        # MDP definition
        self.nS= 4 # Number of MDP states
        self.nA= 2 # Number of MDP actions
        self.nR= 4 # Number of possible rewards

        # # Transition function
        self.T= np.zeros((self.nS, self.nA, self.nS))
        self.T[0, 0, 1]= 0.6
        self.T[0, 0, 2]= 0.4
        self.T[0, 1, 0]= 0.1 #here
        self.T[0, 1, 1]= 0.9 #here
        self.T[1, 0, 0]= 0.2 
        self.T[1, 0, 1]= 0.8
        self.T[1, 1, 2]= 0.7
        self.T[1, 1, 3]= 0.3
        self.T[2, 0, 0]= 0.3
        self.T[2, 0, 2]= 0.7
        self.T[2, 1, 1]= 0.2
        self.T[2, 1, 3]= 0.8
        self.T[3, 0, 2]= 0.6
        self.T[3, 0, 3]= 0.4
        self.T[3, 1, 3]= 1.0
        
        # Reward function
        self.R= np.array([10, -5, 1, -10])
        
        # Current state and step
        self.currentState= None
        self.currentStep= None
        
    
    # returns the number of states in the environment
    def numberOfStates(self):
        return self.nS
       
    # returns the number of states in the environment
    def numberOfActions(self):
        return self.nA
    
    # Returns the transition probability from state s and action a to state sp
    def transitionProb(self, s, a, sp):
        return self.T[s, a, sp]
    
    # Returns the reward value for the transition (s,a)->sp, i.e. r(s,a,sp)
    def rewardValue(self, s, a, sp):
        return self.R[sp]
    
    # Resets the environment to default state
    # Returns the current env. state
    def reset(self):
        # Set default env state and step number
        self.currentState= 0
        self.currentStep= 0
        return int(self.currentState)

    def StoppingCriterionSatisfied(self):
        return self.currentStep >= self.MaxSteps

In [ ]:
# Quantum Environment for Two State Transitions 
# 25-bits classical register
# # Reward qubits stored to ancillary qubits before mid-circuit measurment

class QuantumToyEnv(ClassicToyEnv):
    
    # Constructor. 
    # Param MaxSteps: Maximum number of environment/agent interactions allowed for an episode
    def __init__(self, MaxSteps=np.inf):
        super().__init__(MaxSteps)
    
        # ========================================
        # Quantum MDP representation 
        self.nqS= 2 # Number of qubits for state representation= log2(nS)
        self.nqA= 1 # Number of qubits required for action representation= log2(nA)
        self.nqR= 2 # Number of qubits for reward representation= log2(nR)
        self.nqRt = 4 # Number of qubits for return
        self.nqAnci = 2 # Number of ancillary qubits

        # Quantum Registers for MDP 2-state transition 
        self.qS= QuantumRegister(self.nqS, name='qState') # Quantum Register to store input environment states
        self.qSp= QuantumRegister(self.nqS, name='qNextState') # Quantum Register to store output environment states
        self.qA= QuantumRegister(self.nqA, name='qAction') # Quantum Register to store actions
        self.qR= QuantumRegister(self.nqR, name='qReward') # Quantum Register to store rewards

        self.t0_qAnci = QuantumRegister(self.nqAnci, name = 't0_qAnci') # ancillary qubits to store the timestep 0 reward
        self.t1_qAnci = QuantumRegister(self.nqAnci, name = 't1_qAnci') # ancillary qubits to store the timestep 1 reward
        
        # Quantum Registers for Return
        self.qRt= QuantumRegister(self.nqRt, name='qRt') # Quantum Register to store return
        
        # Classical Registers
        self.cReg= ClassicalRegister(25, name='cReg') # Classical Register to store measured three timesteps and return
        # self.cRt= ClassicalRegister(4, name='cR') # Classical Register to store measured return
        
        # Quantum Circuit to store a cycle of an MDP
        self.qc= QuantumCircuit(self.qS, self.qA, self.qSp, self.qR, self.t0_qAnci,  self.t1_qAnci, self.qRt, self.cReg)
        # self.qc= QuantumCircuit(self.qS, self.qA, self.qSp, self.qR, self.t0_cReg, self.t1_cReg, self.t2_cReg)

        # ----------------------------------------------------------------------------------------------------------------
        # for t = 0 , Initial step      
        # Us implementation: Appends Us to self.qc
        self.__Us_0(self.qc, self.qS)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA)

        # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS, self.qA, self.qSp)
                
        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp, self.qR)
        
        # Uanci implementation: Store the reward at the ancillary qubits
        self.__Uanci(self.qc, self.qR, self.t0_qAnci) 
        
        self.qc.barrier()

        """ Mid-ciruit Measurement at timestep 0 """
        # ====> Mid-ciruit Measurement for timestep 0 
        self.qc.measure([self.qS[0], self.qS[1], self.qA[0], self.qSp[0] ,self.qSp[1], self.qR[0], self.qR[1]], 
                               [self.cReg[0], self.cReg[1], self.cReg[2], self.cReg[3], self.cReg[4], self.cReg[5], self.cReg[6]])
        
        self.qc.barrier()
               
        # -------------------------------------------------------------------------------------------------------------
        # Insert delyas after mid-circuit measurement before reusing qubits
        delay_duration = 2000 # in nanoseconds
        self.qc.delay(delay_duration, self.qS[0], unit= 'ns')
        self.qc.delay(delay_duration, self.qS[1], unit= 'ns')
        self.qc.delay(delay_duration, self.qA[0], unit= 'ns')
        self.qc.delay(delay_duration, self.qR[0], unit= 'ns')
        self.qc.delay(delay_duration, self.qR[1], unit= 'ns')     
                
        # ------------------------------------------------------------------------------------------------------------
        # for t = 1     
        """ Mid-ciruit Reset for 'State qubit', 'Action qubit', 'Reward qubit'. """
        self.qc.reset(self.qS)
        self.qc.reset(self.qA)
        self.qc.reset(self.qR)
        
        """ State transition from t = 0 to t = 1. """       
        # Us implementation: Appends Us to self.qc
        self.__Us(self.qc, self.qSp, self.qS)
       
       # Insert delyas after mid-circuit measurement before resusing qubits
        self.qc.delay(delay_duration, self.qSp[0], unit= 'ns')
        self.qc.delay(delay_duration, self.qSp[1], unit= 'ns')
        
        """ Mid-ciruit Reset for 'Next state'."""
        self.qc.reset(self.qSp)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA)

        # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS, self.qA, self.qSp)
                
        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp, self.qR)
          
        # Uanci implementation: Store the reward at the ancillary qubits
        self.__Uanci(self.qc, self.qR, self.t1_qAnci) 
        
        self.qc.barrier()
         
        """ Mid-ciruit Measurement at timestep 1 """
        # ====> Mid-ciruit Measurement for timestep 0 
        self.qc.measure([self.qS[0], self.qS[1], self.qA[0], self.qSp[0] ,self.qSp[1], self.qR[0], self.qR[1]], 
                               [self.cReg[7], self.cReg[8], self.cReg[9], self.cReg[10], self.cReg[11], self.cReg[12], self.cReg[13]])
        
        self.qc.barrier()

        # -------------------------------------------------------------------------------------------------------------
        # Insert delyas after mid-circuit measurement before resusing qubits
        self.qc.delay(delay_duration, self.qS[0], unit= 'ns')
        self.qc.delay(delay_duration, self.qS[1], unit= 'ns')
        self.qc.delay(delay_duration, self.qA[0], unit= 'ns')
        self.qc.delay(delay_duration, self.qR[0], unit= 'ns')
        self.qc.delay(delay_duration, self.qR[1], unit= 'ns')
        
        # -----------------------------------------------------------------------------------------------------------------
        # for t = 2
        """ Mid-ciruit Reset for 'State qubit', 'Action qubit', 'Reward qubit'. """
        self.qc.reset(self.qS)
        self.qc.reset(self.qA)
        self.qc.reset(self.qR)
        
        """ State transition from t = 1 to t = 2. """    
        # Us implementation: Appends Us to self.qc
        self.__Us(self.qc, self.qSp, self.qS)

        # Insert delyas after mid-circuit measurement before resusing qubits
        self.qc.delay(delay_duration, self.qSp[0], unit= 'ns')
        self.qc.delay(delay_duration, self.qSp[1], unit= 'ns')

        """ Mid-ciruit Reset for 'Next state'."""
        self.qc.reset(self.qSp)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA)

        # # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS, self.qA, self.qSp)

        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp, self.qR)
        
        self.qc.barrier()
               
        # -----------------------------------------------------------------------------------------------------------------
        # Return
        self.__UReturn(self.qc, self.t0_qAnci, self.t1_qAnci, self.qR, self.qRt)
        
        """ Mid-ciruit Measurement at timestep 2 """
        # ====> Mid-ciruit Measurement for timestep 0 
        self.qc.measure([self.qS[0], self.qS[1], self.qA[0], self.qSp[0] ,self.qSp[1], self.qR[0], self.qR[1]], 
                               [self.cReg[14], self.cReg[15], self.cReg[16], self.cReg[17], self.cReg[18], self.cReg[19], self.cReg[20]])       
        
        self.qc.barrier()
        
        self.qc.measure([self.qRt[0], self.qRt[1], self.qRt[2], self.qRt[3]], 
                               [self.cReg[21], self.cReg[22], self.cReg[23], self.cReg[24]])         
        
        
    # Implementation of the QSample encoding
    def __QSampleNextState(self, probs, qc, qS, qA, qSp, control_s, control_a, control_sp= '', currentQ=0):
        
        all_controls= control_s + control_a + control_sp
        
        # Find control qubits
        controlsS= []
        for i in range(len(qS)):
            controlsS.append(qS[i])
        controlsA= []
        for i in range(len(qA)):
            controlsA.append(qA[i])
        controlsSp= []
        for i in range(currentQ):
            controlsSp.append(qSp[i])
        
        # Base case: End of recursion
        if currentQ>=len(qSp):
            return
        
        else: # General case
        
            # compute prob(qS[currentQ | qS, qA, qS[:currentQ-1]])
            add_prob= np.sum(probs)
            if add_prob > 0:
                current_prob= np.sum(probs[len(probs)//2:])/add_prob
                angle= 2*np.arcsin(np.sqrt(current_prob))
                
                # Make custom controlled CRY gate 
                cry= RYGate(angle).control(num_ctrl_qubits=len(all_controls), 
                                             ctrl_state= all_controls)
                
                qc.append(cry, [*controlsSp, *controlsA, *controlsS, qSp[currentQ]])
        
            self.__QSampleNextState(probs[:len(probs)//2], qc, qS, qA, qSp, 
                    control_s= control_s,
                    control_a= control_a,
                    control_sp=control_sp+'0', currentQ=currentQ+1)
            self.__QSampleNextState(probs[len(probs)//2:], qc, qS, qA, qSp, 
                    control_s= control_s,
                    control_a= control_a,
                    control_sp=control_sp+'1', currentQ=currentQ+1)
    

    # Circuit to set input environment state  
    # for state in superposition
    def __Us_0(self, qc, qS):
        for i in range(len(qS)):
            qc.h(qS[i])

    def __Us(self, qc, qSp, qS):   
        for i in range(len(qS)):
           if i == 1:
             qc.cx(qSp[i],qS[i-1])
           else:
            qc.cx(qSp[i],qS[i+1])
       
    # Circuit to set input action    
    def __Ua(self, qc, qA):
        for i in range(len(qA)):
            qc.h(qA[i])
    
    # Circuit to implement transition function
    def __Ut(self, qc, qS, qA, qSp):
        
        for s in range(self.nS): 
            for a in range(self.nA):
                
                # Conditionals for input state s
                binS= bin(s)[2:][::-1]
                while len(binS)<self.nqS:
                    binS+= '0'
                binS= binS[::-1]
                
                # Conditionals for input action a
                binA= bin(a)[2:][::-1]
                while len(binA)<self.nqA:
                    binA+= '0'
                binA= binA[::-1]
        
                # Transition Function P(s,a)
                probs= self.T[s, a]
                self.__QSampleNextState(probs, self.qc, qS, qA, qSp, 
                        control_s=binS, 
                        control_a=binA)

    
    # Circuit to implement reward function
    def __Ur(self, qc, qSp, qR):
        for i in range(len(qSp)):
            qc.cx(qSp[i], qR[i])

    # Circuit to store the reward at the ancillary qubits
    def __Uanci(self, qc, qR, qAnci):
        for i in range(len(qR)):
            qc.cx(qR[i], qAnci[i]) 

    # Circuit to store the reward at the ancillary qubits
    def __UReturn(self, qc, t0_qReward, t1_qReward, qReward, qReturn):
      ## For Return qubit[0]
      qc.cx(t0_qReward[1], qReturn[0])
      qc.cx(t1_qReward[1], qReturn[0])
      qc.cx(qReward[1], qReturn[0])
      
      qc.barrier()

      ## For Return qubit[1]
      qc.cx(t0_qReward[0], qReturn[1])
      qc.cx(t1_qReward[0], qReturn[1])
      qc.cx(qReward[0], qReturn[1])
      # For carry-in part
      qc.ccx(t0_qReward[1], t1_qReward[1], qReturn[1])
      qc.ccx(t0_qReward[1], qReward[1], qReturn[1])
      qc.ccx(t1_qReward[1], qReward[1], qReturn[1])
      qc.barrier()
      
      # For Return qubit [2]
      # when no carry-in bit from LSB
      qc.ccx(t0_qReward[0],  t1_qReward[0], qReturn[2])
      qc.ccx(t0_qReward[0], qReward[0],qReturn[2])
      qc.ccx(t1_qReward[0], qReward[0] ,qReturn[2])

      # when carry-in bit from LSB
      qc.mcx([t0_qReward[1], t1_qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      ## carry-in and single one qubit
      qc.mcx([t0_qReward[1], t1_qReward[1],t0_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], t1_qReward[1], t1_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], t1_qReward[1], qReward[0]],[qReturn[2]])

      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1],  t1_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1], qReward[0]],[qReturn[2]])

      qc.mcx([t1_qReward[1], qReward[1],t0_qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], t1_qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], qReward[0]],[qReturn[2]])

      qc.mcx([t0_qReward[1], t1_qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0],  qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0],  qReward[0]],[qReturn[2]])

      qc.barrier()

      ### for  Return qubit [3] (carry-out qubit)
      qc.mcx([t0_qReward[1], t1_qReward[1], t0_qReward[0], t1_qReward[0],  qReward[0]],[qReturn[3]])
      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[3]])
      qc.mcx([t1_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[3]])
      qc.barrier()
          

In [ ]:
# Instantiate environment
qMDP_return= QuantumToyEnv() 
qMDP_return.qc.draw('mpl', fold=-1, initial_state=True) # 'fold=-1' ensures no line breaks

In [ ]:
## Before transpilation
qc_noG = qMDP_return.qc

print("=== No-Grover: BEFORE transpilation ===")
print("Qubits:", qc_noG.num_qubits)
print("Depth:", qc_noG.depth())
print("Size:", qc_noG.size())
print("Operations:", qc_noG.count_ops())

In [ ]:
print(qc_noG.count_ops())

In [ ]:
for name, count in qc_noG.count_ops().items():
    print(name, count)

In [ ]:
backends = service.backends()

for backend in backends:
    print(backend.name, backend.num_qubits)

In [ ]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

backend = service.backend("ibm_marrakesh")


# Generate the preset pass manager
pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
pm_qMDP_circuit = pm.run(qMDP_return.qc)

In [ ]:
# Execute using Qiskit Primitives

sampler = Sampler(mode=backend)
sampler.options.default_shots = 75000 
sampler.options.dynamical_decoupling.enable = True  #Enable dynamical decoupling to preserve qubit coherence during idle time
sampler.options.dynamical_decoupling.sequence_type = "XY4" # robust sequence
# sampler.options

In [ ]:
job_qMDP = sampler.run([pm_qMDP_circuit])
print(job_qMDP.job_id())
print(job_qMDP.status())

In [ ]:
job_qMDP.status()

In [ ]:
# job_qMDP.cancel()

In [ ]:
results = job_qMDP.result()
print(results)
print(results[0])

In [ ]:
hardware_counts_mdp_return = results[0].data.cReg.get_counts()
print(len(hardware_counts_mdp_return))

In [ ]:
print ('counts_qMDP_return:',hardware_counts_mdp_return) 

# Function to reverse the four most significant bits
def reverse_qubit_order_msb(value):
    if len(value) >= 4:
        most_significant_four = value[:4]
        rest = value[4:]
        reversed_most_significant_four = most_significant_four[::-1]
        return reversed_most_significant_four + rest
    return value

def correct_qubit_order(value):
     if len(value) >= 21:
         # return
         rt = value[:4] 
         # for t2
         t2 = value[4:11]
         t2_msb = reverse_qubit_order_msb(t2)
         # for t1
         t1 = value[11:18]
         t1_msb = reverse_qubit_order_msb(t1)
         # for t0
         t0 = value[18:]
         t0_msb = reverse_qubit_order_msb(t0)
         return rt+ t2_msb+ t1_msb + t0_msb
         # return rt+ " "+ t2_msb+ " " + t1_msb + " "+ t0_msb
     return value

# Create a new dictionary with modified values
modified_hardware_counts_mdp_return = {correct_qubit_order(key): value for key, value in hardware_counts_mdp_return.items()}
print('modified_counts_qMDP_return:',len(modified_hardware_counts_mdp_return)) 

In [ ]:
print(len(modified_hardware_counts_mdp_return))

In [ ]:
print(modified_hardware_counts_mdp_return.keys())

In [ ]:
modified_counts_qMDP_return_sorted = dict(sorted(modified_hardware_counts_mdp_return.items(), key= lambda item: item[0], reverse= False))

Check with Statevector

In [ ]:
# read saved Statevector

import os
import pickle

file_name = './save_QMDP_Static_Circuit/Two State Transitions with Return Statevectors.pkl'

if os.path.exists(file_name):
 print("...found existing State Vector! Reading it...")
 with open(file_name, "rb") as file:
      svector_3steps_return= pickle.load(file)

print(svector_3steps_return)

In [ ]:
prob_amplitudes, qOutputStates = _state_to_latex_ket(svector_3steps_return.data)
print('Probability amplitude and qsample trajectory:\n',prob_amplitudes,'\n',qOutputStates)

# sorting ascending order for amplitude values
prob_amplitudes_sorted = dict(sorted(prob_amplitudes.items(), key= lambda item: item[1]))
print('prob_amplitudes_sorted:',prob_amplitudes_sorted)

# top 5 values of amplitude
top_5_indices = np.argsort(list(prob_amplitudes.values()))[-5:]
print('top 5 indices:',top_5_indices)

# checking the magnitude of statevector is sum to '1'
sum = 0
for i in prob_amplitudes.values():
    i = abs(i)**2
    sum = sum + i
print('total sum:',sum)

print(len(qOutputStates))

In [ ]:
# Correcting Next state qubits and reward qubits order
print ('qOutputStates:',qOutputStates) 

# Function to reverse the four most significant bits
def reverse_qubit_order_msb(value):
    if len(value) >= 4:
        most_significant_four = value[:4]
        rest = value[4:]
        reversed_most_significant_four = most_significant_four[::-1]
        return reversed_most_significant_four + rest
    return value

def correct_qubit_order(value):
     if len(value) >= 21:
         # return
         rt = value[:4] 
         # for t2
         t2 = value[4:11]
         t2_msb = reverse_qubit_order_msb(t2)
         # for t1
         t1 = value[11:18]
         t1_msb = reverse_qubit_order_msb(t1)
         # for t0
         t0 = value[18:]
         t0_msb = reverse_qubit_order_msb(t0)
         return rt+t2_msb+ t1_msb + t0_msb
     return value

# Create a new dictionary with modified values
modified_qOutputStates  = {key: correct_qubit_order(value) for key, value in qOutputStates.items()}
print('modified_qOutputStates:',modified_qOutputStates) ## Next state qubits and reward qubits are in correct order.

In [ ]:
# filter only the valid quantum samples

valid_samples = list(modified_qOutputStates.values()) # Next state qubits and reward qubits' orders are corrected.

filter_valid_modified_qMDP_return= {key: value for key, value in modified_hardware_counts_mdp_return.items() if key in valid_samples}
print(f'Total QSample simulation: {len(filter_valid_modified_qMDP_return)} samples.')
print('QSample simulation_correct qubit order:', filter_valid_modified_qMDP_return)

valid_counts_sorted = dict(sorted(filter_valid_modified_qMDP_return.items(), key= lambda item: item[1], reverse= False))
valid_counts_sorted

In [ ]:
valid_counts_sorted.keys()

Check with reference trajectories 

In [ ]:
# read saved file: the reference 170 trajectories from PRA publication

import os
import pickle
from qiskit.visualization import plot_histogram

file_name = './previous_static_QMDP_data/ Classical Results for Two State Transitions with Return.pkl' 

if os.path.exists(file_name):
 print("...found existing Classical Measurement! Reading it...")
 with open(file_name, "rb") as file:
       results = pickle.load(file)

counts_mdp= results.get_counts()    

print(counts_mdp)


In [ ]:
print ('counts_mdp:',counts_mdp) 

# Function to reverse the four most significant bits
def reverse_qubit_order_msb(value):
    if len(value) >= 4:
        most_significant_four = value[:4]
        rest = value[4:]
        reversed_most_significant_four = most_significant_four[::-1]
        return reversed_most_significant_four + rest
    return value

def correct_qubit_order(value):
     if len(value) >= 21:
         # return
         rt = value[:4] 
         # for t2
         t2 = value[4:11]
         t2_msb = reverse_qubit_order_msb(t2)
         # for t1
         t1 = value[11:18]
         t1_msb = reverse_qubit_order_msb(t1)
         # for t0
         t0 = value[18:]
         t0_msb = reverse_qubit_order_msb(t0)
         return rt+t2_msb+ t1_msb+ t0_msb
     return value

# Create a new dictionary with modified values
modified_counts_mdp = {correct_qubit_order(key): value for key, value in counts_mdp.items()}
print('modified_counts_mdp:',modified_counts_mdp) 

In [ ]:
modified_counts_mdp_sorted = dict(sorted(modified_counts_mdp.items(), key= lambda item: item[0], reverse= False))
modified_counts_mdp_sorted

reference_170_trajectories_with_return = list(modified_counts_mdp_sorted.keys())
len(reference_170_trajectories_with_return)

In [ ]:
filtered_counts_qMDP_return= {key: value for key, value in modified_hardware_counts_mdp_return.items() if key in reference_170_trajectories_with_return}
print('filtered_classical_measurement:',len(filtered_counts_qMDP_return))

filtered_counts_qMDP_return_sorted = dict(sorted(filtered_counts_qMDP_return.items(), key= lambda item: item[0], reverse= False))
filtered_counts_qMDP_return_sorted